In [ ]:
import re
from pprint import pprint

# ==============================
# FUNÇÕES AUXILIARES
# ==============================

def limpar_texto(texto):
    return re.sub(r'\(\d+,\d+\)', '', texto.lower()).strip()

def normalizar_legislacao(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s./§]', '', texto)  # Remove pontuação exceto . / §
    texto = texto.replace("artigo", "art").replace("nº", "").replace("n°", "")
    return texto

# ==============================
# EXTRAÇÃO LEGISLAÇÃO - PERFEITA
# ==============================

def extrair_legislacao(texto):
    texto_norm = texto.lower()
    termos = []

    # Artigos e incisos
    arts = re.findall(r'art\.?\s*(\d+)(?:,\s*inciso\s*([ivxlcdm]+))?', texto_norm)
    for art in arts:
        termos.append(f"art {art[0]}")
        if len(art) > 1 and art[1]:
            termos.append(f"inciso {art[1]}")

    # CRFB
    if any(x in texto_norm for x in ["crfb", "constituição"]):
        termos.append("crfb")

    # Súmulas
    if "sumula" in texto_norm or "súmula" in texto_norm:
        nums = re.findall(r"\d+", texto_norm)
        for n in nums:
            termos.append(f"sumula {n}")

    return list(set(termos))

# ==============================
# EXTRAÇÃO SUBITENS - CORRIGIDA
# ==============================

def extrair_subitens(texto):
    texto = re.sub(r'\s*Peso:\s*\d+(?:,\d+)?', '', texto, flags=re.IGNORECASE)
    pieces = re.split(r'(\(.*?\))', texto)
    pattern = r'(\d+,\d+)'
    subitens = []

    for i in range(0, len(pieces)-1, 2):
        texto_antes = pieces[i].strip()
        if i+1 < len(pieces) and pieces[i+1].strip():
            peso_match = re.search(pattern, pieces[i+1])
            if peso_match:
                peso = float(peso_match.group(1).replace(',', '.'))
                subitens.append({
                    "texto": texto_antes,
                    "peso": peso
                })

    return subitens

# use o resto do seu código (extrair_componentes, transformar_gabarito etc.) normalmente

# ==============================
# CLASSIFICAÇÃO E COMPONENTES
# ==============================

def classificar_componente(texto):
    texto_norm = texto.lower()
    keywords = ["art", "lei", "crfb", "sumula", "súmula", "cf", "constituição"]
    return "legislacao" if any(k in texto_norm for k in keywords) else "semantico"


def extrair_componentes(texto):
    subitens = extrair_subitens(texto)
    componentes = []

    for sub in subitens:
        tipo = classificar_componente(sub["texto"])
        if tipo == "legislacao":
            componentes.append({
                "tipo": "legislacao",
                "termos": extrair_legislacao(sub["texto"]),
                "peso": sub["peso"]
            })
        else:
            componentes.append({
                "tipo": "semantico",
                "referencia": limpar_texto(sub["texto"]),
                "peso": sub["peso"]
            })
    return componentes

# ==============================
# FUNÇÃO PRINCIPAL
# ==============================

def extrair_secao(linha):
    m = re.match(r'^\[(.*?)\]', linha)
    return m.group(1).strip() if m else None

def extrair_id(linha):
    m = re.search(r'([A-Z]\d*|\d+(?:\.\d+)?)\.', linha)
    return m.group(1).strip() if m else None

def transformar_gabarito(texto):
    criterios = []
    for linha in texto.split('\n'):
        linha = linha.strip()
        if not linha or '(' not in linha: continue

        item_id = extrair_id(linha)
        if not item_id: continue

        secao = extrair_secao(linha)
        conteudo = re.sub(r'^\[.*?\]\s*', '', linha)
        conteudo = re.sub(rf'^{re.escape(item_id)}\.\s*', '', conteudo).strip()

        componentes = extrair_componentes(conteudo)
        if componentes:
            criterios.append({
                "id": item_id,
                "secao": secao,
                "peso_total": round(sum(c["peso"] for c in componentes), 2),
                "componentes": componentes
            })
    return {"criterios": criterios}

# ==============================
# 🧪 TESTE
# ==============================

# Seu gabarito original
gabarito = """
Itens de Correção:
[Endereçamento]1. A peça deve ser endereçada ao Juízo da Vara Cível ou da Vara de Fazenda Pública da Comarca do Município Alfa (0,10).Peso: 0.1
[Qualificação das partes]2. Impetrante: sociedade empresária XYZ (0,10).Peso: 0.3
[Qualificação das partes]3. Autoridade coatora: João da Silva/Presidente da Comissão de Licitação (0,10).
[Qualificação das partes]4. Pessoa jurídica interessada: Município Alfa (0,10).
[Cabimento]5. Houve violação a direito líquido e certo (0,10), nos termos do Art. 5º, inciso LXIX, da CRFB/88, ou do Art. 1º da Lei nº 12.016/2009 (0,10),Peso: 0.2
[Cabimento]5.1 Observância do prazo decadencial de 120 dias (0,10), na forma do Art. 23 da Lei nº 12.016/2009 (0,10).Peso: 0.2
[Fundamentação]6. O prazo mínimo para apresentação de propostas, contados a partir da data de divulgação do edital de licitação, é de 08 (oito) dias úteis, quando adotado o critério de julgamento de menor preço (0,50), nos termos do Art. 55, inciso I, alínea a, da Lei nº 14.133/2021 (0,10).Peso: 0.6
[Fundamentação]7. A garantia de proposta, como requisito de pré-habilitação, não pode ser superior a 1% (um por cento) do valor estimado para a contratação (0,50), na forma do Art. 58, §1º, da Lei nº 14.133/2021 (0,10).Peso: 0.6
[Fundamentação]8. Não há margem de preferência para sociedades empresárias que tenham instituído programa de integridade (0,50), por ausência de previsão no Art. 26 da Lei nº 14.133/2021 (0,10).Peso: 0.6
[Fundamentação]9. Não se pode exigir do licitante valores mínimos de faturamento anterior e de índices de rentabilidade ou lucratividade para demonstração da sua aptidão econômica para cumprir as obrigações decorrentes do futuro contrato (0,50), conforme prevê o Art. 69, §2º, da Lei nº 14.133/2021 (0,10).Peso: 0.6
[Fundamentação]10. Em caso de empate entre duas ou mais propostas, a Administração Pública não tem qualquer discricionariedade na escolha do licitante vencedor, estando vinculada aos critérios de desempate legalmente estabelecidos (0,50), na forma do Art. 60 da Lei nº 14.133/2021 (0,10).Peso: 0.6
[Fundamentos para a concessão da medida liminar]11. A probabilidade do direito está presente e decorre da violação dos dispositivos legais elencados na fundamentação (0,15).Peso: 0.15
[Fundamentos para a concessão da medida liminar]12. Há inequívoco e fundado receio de ineficácia da medida caso seja concedida a segurança apenas ao final do processo, porquanto a licitação prosseguirá com base em edital maculado, em prejuízo da impetrante (0,15).Peso: 0.15
[Pedidos]13. Concessão de medida liminar para suspender o prosseguimento do processo licitatório, até o julgamento do mérito do mandado de segurança (0,10), na forma do Art. 7º, inciso III, da Lei nº 12.016/2009 (0,10).Peso: 0.2
[Pedidos]14. Notificação da autoridade coatora (0,10)Peso: 0.1
[Pedidos]15. Ciência ao órgão de representação judicial da pessoa jurídica de direito público a que está vinculada (0,10).Peso: 0.1
[Pedidos]16. Juntada de documentos que demonstrem a prova pré-constituída (0,10).Peso: 0.1
[Pedidos]17. Concessão da segurança, com a confirmação da medida liminar concedida (0,10), para anular o edital da licitação e todos os atos, no âmbito do processo licitatório, que lhe são subsequentes (0,10).Peso: 0.2
[Fechamento]18. Valor da causa (0,10).Peso: 0.1
[Fechamento]19. Local, data, advogado e inscrição OAB (0,10).Peso: 0.1
"""

resultado = transformar_gabarito(gabarito)
pprint("Gabarito:")
pprint(gabarito)
print("Resultado:")
pprint(resultado)